In [70]:
import json
def evaluate(file_path,ground_truth="Non-binary"):
    with open(file_path, 'r') as f:
        qa_results = json.load(f)
    
    # Calculate statistics
    total = len(qa_results)
    yes = sum(1 for qa in qa_results if qa['final_answer']=="Yes")
    no = sum(1 for qa in qa_results if qa['final_answer']=="No")
    retrieval_collapse = sum(1 for qa in qa_results if qa['final_answer']=='Not enough information')
    
    stats = {"Yes": yes, "No": no, "Not enough information": retrieval_collapse}

    # Calculate RAG performance metrics
    retrieval_success = total - retrieval_collapse # = yes + no

    rag_scores = {
        # the retrieved context is either none or insufficient to answer the question, leading to "Not enough information" answer
        "failure_rate": retrieval_collapse / total if total > 0 else 0, 
        # there is sufficient retrieved context leading to a "Yes" or "No" answer, but the final answer may still be incorrect due to reasoning errors from zero-shot classification
        "success_rate": retrieval_success / total if total > 0 else 0  
    }

    if ground_truth == "Yes": 
        # true positive is "Yes", false negative is "No" or "Not enough information", and no false positive
        # recall = accuracy = tp / (tp + fn) = true positve rate
        false_negative_cases = no + retrieval_collapse
        recall = yes / (yes + false_negative_cases) if (false_negative_cases) > 0 else 0
        precision = 1 # precision = TP / (TP + FP), and FP = 0 in this case
        
        
        tp_rate = yes / total if total > 0 else 0
        fn_rate = false_negative_cases / total if total > 0 else 0

        f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0 
        metrics = {"precision": precision, "recall": recall,"true_positive_rate": tp_rate, "false_negative_rate": fn_rate, "f1_score": f1_score}
    
    elif ground_truth == "No": 
        # true positive is "No", false negative is "Yes" or "Not enough information", and no false positive
        # recall = accuracy = tp / (tp + fn) = true positve rate
        false_negative_cases = yes + retrieval_collapse
        recall = no / (no + false_negative_cases) if (no + false_negative_cases) > 0 else 0
        precision = 1 # precision = TP / (TP + FP), and FP = 0 in this case

        tp_rate = no / total if total > 0 else 0
        fn_rate = false_negative_cases / total if total > 0 else 0
        f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0 

        metrics = {"precision": precision, "recall": recall,"true_positive_rate": tp_rate, "false_negative_rate": fn_rate, "f1_score": f1_score}
    elif ground_truth == "Non-binary":
        # Case of PubMedQA_labeled where the ground truth is non-binary
        # true positive = qa['final_answer'] matches ground truth final_decision 
        # false negative = "Not enough information" when the ground truth is "yes" or "no"
        # and false positive = "not enough information" when the ground truth is "maybe"
        with open('../pqa_labeled_mapping.json', 'r') as f:
            ground_truth_mapping = json.load(f)

        # mapping questions to their corresponding ground truth final_decision
        question_to_decision = {item['question']: item['final_decision'] for item in ground_truth_mapping}
        
        true_positives = sum(1 for qa in qa_results if qa['final_answer'].lower()==question_to_decision.get(qa['question'], "Unknown"))
        true_negatives = sum(1 for qa in qa_results if qa['final_answer']=='Not enough information' and question_to_decision.get(qa['question'], "Unknown")=='maybe')
        
        false_negatives = sum(1 for qa in qa_results if qa['final_answer'].lower()=='Not enough information' and question_to_decision.get(qa['question'], "Unknown")!='maybe')
        false_positives = sum(1 for qa in qa_results if qa['final_answer'].lower()!='Not enough information' and question_to_decision.get(qa['question'], "Unknown")=='maybe')
        
        precision = {"Yes/No":true_positives/(true_positives + false_positives), "Not enough information": true_negatives/(true_negatives + false_negatives)}
        recall = {"Yes/No":true_positives/(true_positives + false_negatives), "Not enough information": true_negatives/(true_negatives + false_positives)}
        
        macro_true_positives = (true_positives + true_negatives)
        macro_false_positives = (false_positives + false_negatives) # = macro_false_negatives (i.e misclassification between "Not enough information" and "Yes/No")
        stats = {"TP": true_positives, "TN": true_negatives, "FP": false_positives, "FN": false_negatives}
        
        # precision = recall
        macro_precision = macro_true_positives / (macro_true_positives + macro_false_positives) if (macro_true_positives + macro_false_positives) > 0 else 0
        macro_recall = macro_precision # macro_precision = macro_recall in this case since false positives = false negatives
        tp_rate = macro_true_positives / total if total > 0 else 0
        fn_rate = macro_false_positives / total if total > 0 else 0

    
        f1_score = 2 * (macro_precision * macro_recall) / (macro_precision + macro_recall) if (macro_precision + macro_recall) > 0 else 0 

        metrics = {"macro_precision": macro_precision, "macro_recall": macro_recall,"true_positive_rate": tp_rate, "false_negative_rate": fn_rate, "f1_score": f1_score}
    

    return stats, rag_scores, metrics

In [81]:
def confusion_matrix(stats,ground_truth="Non-binary"):
    if ground_truth == "Yes":
        tp = stats["Yes"]
        fn = stats["No"] + stats["Not enough information"]
        fp = 0
        tn = 0
    elif ground_truth == "No":
        tp = stats["No"]
        fn = stats["Yes"] + stats["Not enough information"]
        fp = 0
        tn = 0
    elif ground_truth == "Non-binary":
        tp = stats["TP"]
        fn = stats["FN"]
        fp = stats["FP"]
        tn = stats["TN"]

    return {"TP": tp, "FN": fn, "FP": fp, "TN": tn}

def print_confusion_matrix(conf_matrix):
    print("Confusion Matrix:")
    print("\t TP | FN")
    print("-"*20)
    print(f"TP\t {conf_matrix['TP']} | {conf_matrix['FN']}  | FN")
    print(f"FP\t  {conf_matrix['FP']} |  {conf_matrix['TN']}  | TN")
    print("-"*20)
    

In [78]:
def print_evaluation(stats, rag_scores, metrics,ground_truth="Non-binary"):
    print("Evaluation Results:")
    if ground_truth != "Non-binary":
        
        print(f"Total Questions: {sum(stats.values())}")
        print(f"Yes: {stats['Yes']}, No: {stats['No']}, Not enough information: {stats['Not enough information']}")
        
        print("\nRAG Performance Metrics:")
        print(f"Failure Rate (Not enough information): {rag_scores['failure_rate']:.2%}")
        print(f"Success Rate (Yes/No): {rag_scores['success_rate']:.2%}")
        
        print("\nPrecision and Recall:")
        print(f"Precision: {metrics['precision']:.2%}")
        print(f"Recall: {metrics['recall']:.2%}")
        print(f"F1 Score: {metrics['f1_score']:.2%}")
        print(f"True Positive Rate: {metrics['true_positive_rate']:.2%}")
        print(f"False Negative Rate: {metrics['false_negative_rate']:.2%}")
    else:
        print(f"Total Questions: {sum(stats.values())}")
        print(f"Yes/No: {stats['TP']+stats['FN']}, Not enough information: {stats['TN']+stats['FP']}")
        
        print("\nRAG Performance Metrics:")
        print(f"Failure Rate (Not enough information): {rag_scores['failure_rate']:.2%}")
        print(f"Success Rate (Yes/No): {rag_scores['success_rate']:.2%}")
        
        print("\nPrecision and Recall:")
        print(f"Precision: {metrics['macro_precision']:.2%}")
        print(f"Recall: {metrics['macro_recall']:.2%}")
        print(f"F1 Score: {metrics['f1_score']:.2%}")
        print(f"True Positive Rate: {metrics['true_positive_rate']:.2%}")
        print(f"False Negative Rate: {metrics['false_negative_rate']:.2%}")

In [82]:
file_path = "noQA_3R.json"
no_stats, no_rag_scores, no_metrics = evaluate(file_path, ground_truth="No")
print_confusion_matrix(confusion_matrix(no_stats, ground_truth="No"))
print_evaluation(no_stats, no_rag_scores, no_metrics,"No")

Confusion Matrix:
	 TP | FN
--------------------
TP	 20 | 73  | FN
FP	  0 |  0  | TN
--------------------
Evaluation Results:
Total Questions: 93
Yes: 5, No: 20, Not enough information: 68

RAG Performance Metrics:
Failure Rate (Not enough information): 73.12%
Success Rate (Yes/No): 26.88%

Precision and Recall:
Precision: 100.00%
Recall: 21.51%
F1 Score: 35.40%
True Positive Rate: 21.51%
False Negative Rate: 78.49%


In [83]:
file_path = "yesQA_3R.json"
yes_stats, yes_rag_scores, yes_metrics = evaluate(file_path, ground_truth="Yes")
print_confusion_matrix(confusion_matrix(yes_stats, ground_truth="Yes"))
print_evaluation(yes_stats, yes_rag_scores, yes_metrics,"Yes")

Confusion Matrix:
	 TP | FN
--------------------
TP	 9 | 84  | FN
FP	  0 |  0  | TN
--------------------
Evaluation Results:
Total Questions: 93
Yes: 9, No: 4, Not enough information: 80

RAG Performance Metrics:
Failure Rate (Not enough information): 86.02%
Success Rate (Yes/No): 13.98%

Precision and Recall:
Precision: 100.00%
Recall: 9.68%
F1 Score: 17.65%
True Positive Rate: 9.68%
False Negative Rate: 90.32%


In [84]:
file_path = "pqa_labeled_3R.json"
stats, rag_scores, metrics = evaluate(file_path, ground_truth="Non-binary")
print_confusion_matrix(confusion_matrix(stats, ground_truth="Non-binary"))
print_evaluation(stats, rag_scores, metrics,"Non-binary")

Confusion Matrix:
	 TP | FN
--------------------
TP	 5 | 0  | FN
FP	  110 |  108  | TN
--------------------
Evaluation Results:
Total Questions: 223
Yes/No: 5, Not enough information: 218

RAG Performance Metrics:
Failure Rate (Not enough information): 97.30%
Success Rate (Yes/No): 2.70%

Precision and Recall:
Precision: 50.67%
Recall: 50.67%
F1 Score: 50.67%
True Positive Rate: 11.30%
False Negative Rate: 11.00%


### IMPORT PUBMEDQA BENCHMARK FOR EVALUATION

In [16]:
import pandas as pd

df = pd.read_parquet("hf://datasets/qiaojin/PubMedQA/pqa_labeled/train-00000-of-00001.parquet")
df.columns
print(df['final_decision'].value_counts())

pubmedQA_set = df.to_dict(orient='records')

final_decision
yes      552
no       338
maybe    110
Name: count, dtype: int64


In [17]:
import numpy as np
def convert(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    raise TypeError(f"Object of type {type(obj)} is not JSON serializable")

with open("../pqa_labeled.json",'w') as f:
    json.dump(pubmedQA_set, f,default=convert, indent=4)

In [21]:
with open('../pqa_labeled.json', 'r') as f:
    pubmedQA_set = json.load(f)

# Create a mapping from final_answer to final_decision in the PubMedQA dataset
ground_truth_table = []
for qa in pubmedQA_set:
    ground_truth_table.append({
        "question": qa['question'],
        "final_decision": qa['final_decision']
    })
with open("../pqa_labeled_mapping.json",'w') as f:
    json.dump(ground_truth_table, f,default=convert, indent=4)

In [65]:
with open('../pqa_labeled_mapping.json', 'r') as f:
    ground_truth_mapping = json.load(f)
with open('pqa_labeled_3R.json', 'r') as f:
    qa_results = json.load(f)
question_to_decision = {item['question']: item['final_decision'] for item in ground_truth_mapping}

for qa in qa_results:
    final_answer = qa['final_answer'].lower()
    corresponding_decision = question_to_decision.get(qa['question'], "Unknown")
    print(final_answer +" vs " +corresponding_decision)

not enough information vs yes
no vs no
not enough information vs yes
not enough information vs no
not enough information vs yes
not enough information vs yes
not enough information vs maybe
not enough information vs no
not enough information vs no
not enough information vs yes
not enough information vs yes
not enough information vs no
not enough information vs yes
not enough information vs no
not enough information vs yes
no vs yes
not enough information vs yes
not enough information vs yes
not enough information vs yes
not enough information vs yes
not enough information vs yes
not enough information vs yes
not enough information vs yes
not enough information vs yes
not enough information vs yes
not enough information vs no
not enough information vs yes
not enough information vs maybe
not enough information vs yes
not enough information vs yes
not enough information vs no
not enough information vs maybe
not enough information vs no
not enough information vs yes
not enough information 